# MedVision Thesis - Part 10: GACR-B Ablation (3 seeds)

Trains the **GACR-B** ablation variant on 3 seeds (42, 123, 456). This variant tests whether using **G**ate-**A**ware **C**lass **R**ebalancing (B variant) recovers LMH-like gains through a different mechanism.

| Item | Value |
|------|-------|
| Estimated time | ~7.5 hours |
| Seeds | [42, 123, 456] (ABLATION_SEED_LIST) |

## Setup
1. **Inputs**: Attach Part 1's output AND Part 9's output (Kaggle dataset)
2. **Settings**: Enable Internet + GPU T4 x2
3. **Run All** (~7.5 hours)
4. **Save Version > Quick Save** when done
5. Open `medvision-thesis-final.ipynb`

## Resume
- Already-trained seeds are **skipped automatically** (via `all_results.pkl`)
- If the session times out, re-run the same notebook — it picks up where it left off


In [1]:
!pip install -q transformers==4.44.2 datasets torchvision scikit-learn matplotlib seaborn nltk tqdm statsmodels nbformat requests torchxrayvision
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
import importlib
for pkg in ['transformers', 'datasets', 'statsmodels', 'torchxrayvision']:
    try: importlib.import_module(pkg); print(f"  [OK] {pkg}")
    except ImportError: raise ImportError(f"Package '{pkg}' failed to install.")
print("All dependencies installed.")
import torch
if torch.cuda.is_available(): print(f"  GPU: {torch.cuda.get_device_name(0)}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 571.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.9 MB/s eta 0:00:00
  [OK] transformers
  [OK] datasets
  [OK] statsmodels
  [OK] torchxrayvision
All dependencies installed.
  GPU: Tesla T4


In [2]:
import os, sys, json, random, re, copy, time, warnings, math, io, shutil, gc, hashlib, zipfile, pickle, subprocess
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
import torchvision
from torchvision import transforms
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report, brier_score_loss)
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import stats as scipy_stats
from scipy.stats import wilcoxon, ttest_rel
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import requests
import torchxrayvision as xrv
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
def mem_stats(label=""):
    ram_mb = 0
    try:
        import resource; ram_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    except: pass
    gpu_mb = torch.cuda.memory_allocated() / 1e6 if torch.cuda.is_available() else 0
    if label: print(f"  [mem {label}] RAM={ram_mb:.0f}MB, GPU={gpu_mb:.0f}MB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torchxrayvision: {xrv.__version__}")
print("Imports OK.")


PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
torchxrayvision: 1.5.2
Imports OK.


In [3]:
TB_DECISION = 'drop'
PHASE0_FREEZE_LMH = True
PHASE0_ACCEPT_NEGATIVE = True
class Config:
    SEED = 42
    SEED_LIST = [42, 123, 456, 789, 1010, 1111, 1212, 1313]
    ABLATION_SEED_LIST = [42, 123, 456]
    BASELINE_EPOCHS = 10
    LMH_EPOCHS = 10
    GACR_B_EPOCHS = 10
    NOISE_CONSISTENCY_EPOCHS = 10
    ADAPTIVE_CADQ_EPOCHS = 10
    ABLATION_EPOCHS = 10
    LMH_G_MAX = 5.0
    LMH_LAMBDA_ENTROPY = 0.05
    LMH_WARM_START = True
    LMH_ANNEAL_EPOCHS = 3
    LMH_BIAS_EPSILON = 0.1
    LMH_BIAS_TEMP = 2.0
    LMH_CALIBRATE_BIAS = True
    LMH_G_INIT_BIAS = -2.0
    LMH_ASSERT_LOSS_POSITIVE = True
    LR_OOF_FOLDS = 5
    PHYSIONET_IMAGE_DIR = '/kaggle/working/images'
    FRONTAL_ONLY = True
    NORMAL_CAP = 2500
    PNEUMONIA_CAP = 2500
    PNEUMOTHORAX_CAP = 1000
    TEXT_MAX_LEN = 256
    IMG_SIZE = 224
    PROJECTION_DIM = 512
    NUM_HEADS = 8
    NUM_CLASSES = 3
    LABEL_NAMES = ['Normal', 'Pneumonia', 'Pneumothorax']
    TEXT_ENCODER = 'emilyalsentzer/Bio_ClinicalBERT'
    TEXT_FROZEN_LAYERS = 4
    IMAGE_FROZEN_STEM_ONLY = True
    DROPOUT_RATE = 0.25
    BATCH_SIZE = 8
    GRAD_ACCUM_STEPS = 4
    LR_IMG_BACKBONE = 1e-5
    LR_IMG_HEAD = 5e-5
    LR_TEXT = 1e-5
    LR_FUSION = 1e-4
    LR_GATE = 5e-5
    LR_G_HEAD = 1e-4
    WEIGHT_DECAY = 1e-4
    WARMUP_RATIO = 0.1
    LABEL_SMOOTHING = 0.05
    FOCAL_GAMMA = 1.5
    USE_CLASS_WEIGHTS = True
    GATE_GRAD_CLIP = 10.0
    TB_PER_BATCH = 2
    TB_OVERSAMPLE = 5
    PN_OVERSAMPLE = 3
    CLASS_WEIGHT_BETA = 0.99
    CADQ_MIN_IMAGE_GATE = 0.20
    CADQ_MAX_IMAGE_GATE = 0.80
    GATE_INIT_LOGITS = [-0.5, 0.0, 0.5]
    STOCHASTIC_TEXT_DROP = 0.10
    NOISE_CONSISTENCY_LAMBDA = 0.3
    NOISE_CONSISTENCY_WARMUP = 2
    SAVE_DIR = '/kaggle/working/checkpoints'
    RESULTS_DIR = '/kaggle/working/results'
cfg = Config()
for d in [cfg.SAVE_DIR, cfg.RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(cfg.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Device: {device}")


  Device: cuda


In [4]:
# Load Part 1 output (data + images + split)
print("=" * 70)
print("  LOADING PART 1 OUTPUT")
print("=" * 70)
PART1_DIR = None
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'mimic_df.pkl' in files:
            PART1_DIR = root; break
if PART1_DIR is None:
    raise FileNotFoundError("Part 1 output not found. Attach Part 1 output as input.")
print(f"  [FOUND] Part 1 output at: {PART1_DIR}")

# Find images directory
IMAGES_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    jpg_count = len([f for f in files if f.endswith('.jpg')])
    if jpg_count > 100:
        IMAGES_DIR = root
        print(f"  [FOUND] Images at: {IMAGES_DIR} ({jpg_count} jpgs)")
        break
# Also check /kaggle/working/images
if not IMAGES_DIR and os.path.exists('/kaggle/working/images'):
    jpg_count = len([f for f in os.listdir('/kaggle/working/images') if f.endswith('.jpg')])
    if jpg_count > 100:
        IMAGES_DIR = '/kaggle/working/images'
        print(f"  [FOUND] Images at: {IMAGES_DIR} ({jpg_count} jpgs)")

# Load mimic_df BEFORE fallback download (so iterrows() works)
with open(os.path.join(PART1_DIR, 'mimic_df.pkl'), 'rb') as f:
    mimic_df = pickle.load(f)
# Download images if not found
if not IMAGES_DIR or jpg_count < 100:
    print("  [INFO] Images not found in Part 1 output. Downloading from PhysioNet...")
    img_dir = Config.PHYSIONET_IMAGE_DIR
    os.makedirs(img_dir, exist_ok=True)
    user = os.environ.get('PHYSIONET_USER', '')
    pwd = os.environ.get('PHYSIONET_PASS', '')
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        user = secrets.get_secret('PHYSIONET_USER') or user
        pwd = secrets.get_secret('PHYSIONET_PASS') or pwd
    except: pass
    if not user or not pwd:
        from getpass import getpass
        user = input("PhysioNet username: ")
        pwd = getpass("PhysioNet password: ")
    import concurrent.futures
    to_dl = []
    for _, row in mimic_df.iterrows():
        out = os.path.join(img_dir, f"{row['dicom_id']}.jpg")
        if not os.path.exists(out) or os.path.getsize(out) < 1000:
            url = f"https://physionet.org/files/mimic-cxr-jpg/2.1.0/files/p{str(row['subject_id'])[:2]}/p{row['subject_id']}/s{row['study_id']}/{row['dicom_id']}.jpg"
            to_dl.append((url, out))
    print(f"  {len(to_dl)} images to download")
    if to_dl:
        def dl(args):
            url, out = args
            cmd = ['wget', '-q', '-c', '--user=' + user, '--password=' + pwd, '--timeout=30', '--tries=3', '-O', out, url]
            try:
                r = subprocess.run(cmd, capture_output=True, timeout=60)
                return r.returncode == 0 and os.path.exists(out) and os.path.getsize(out) > 1000
            except: return False
        ok = 0
        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
            for i, f in enumerate(concurrent.futures.as_completed({ex.submit(dl, a): a for a in to_dl}), 1):
                if f.result(): ok += 1
                if i % 50 == 0: print(f"    {i}/{len(to_dl)}", flush=True)
        print(f"  Downloaded: {ok}/{len(to_dl)}")
    IMAGES_DIR = img_dir

# Load pickles (mimic_df already loaded above)
print(f"  [LOADED] mimic_df.pkl ({len(mimic_df):,} rows)")
mimic_df['img_path'] = mimic_df.apply(lambda r: os.path.join(IMAGES_DIR, f"{r['dicom_id']}.jpg"), axis=1)

# ============================================================
# VERIFY ALL IMAGES EXIST — remove missing from cohort and split
# ============================================================
print(f"\n  [VERIFY] Checking all {len(mimic_df):,} images...")
missing_mask = ~mimic_df['img_path'].apply(os.path.exists)
n_missing = missing_mask.sum()
if n_missing > 0:
    missing_ids = mimic_df.loc[missing_mask, 'dicom_id'].tolist()
    print(f"  [WARN] {n_missing} images missing! Removing from cohort and split...")
    print(f"  [WARN] Missing dicom_ids: {missing_ids[:10]}{'...' if len(missing_ids) > 10 else ''}")
    mimic_df = mimic_df[~missing_mask].reset_index(drop=True)
    print(f"  [CLEANED] mimic_df now has {len(mimic_df):,} rows (removed {n_missing})")
else:
    print(f"  [OK] All {len(mimic_df):,} images accessible")

# Load EXACT split from Part 1 — and remove missing dicom_ids from split mapping
final_split_path = os.path.join(PART1_DIR, 'final_split.pkl')
if os.path.exists(final_split_path):
    print(f"\n  [LOADING] Part 1's exact split from final_split.pkl")
    with open(final_split_path, 'rb') as f:
        split_mapping = pickle.load(f)
    # Clean split mapping: remove missing dicom_ids
    valid_ids = set(mimic_df['dicom_id'].tolist())
    split_mapping['train'] = [d for d in split_mapping['train'] if d in valid_ids]
    split_mapping['validation'] = [d for d in split_mapping['validation'] if d in valid_ids]
    split_mapping['test'] = [d for d in split_mapping['test'] if d in valid_ids]
    train_df = mimic_df[mimic_df['dicom_id'].isin(split_mapping['train'])].reset_index(drop=True)
    val_df = mimic_df[mimic_df['dicom_id'].isin(split_mapping['validation'])].reset_index(drop=True)
    test_df = mimic_df[mimic_df['dicom_id'].isin(split_mapping['test'])].reset_index(drop=True)
    train_df['split'] = 'train'; val_df['split'] = 'validation'; test_df['split'] = 'test'
    print(f"  [OK] Loaded cleaned split: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
else:
    print(f"\n  [WARN] final_split.pkl not found. Falling back to GroupShuffleSplit...")
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
    train_idx, temp_idx = next(gss1.split(mimic_df, groups=mimic_df['subject_id']))
    temp_df = mimic_df.iloc[temp_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
    rel_val_idx, rel_test_idx = next(gss2.split(temp_df, groups=temp_df['subject_id']))
    train_df = mimic_df.iloc[train_idx].reset_index(drop=True)
    val_df = temp_df.iloc[rel_val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[rel_test_idx].reset_index(drop=True)
    train_df['split'] = 'train'; val_df['split'] = 'validation'; test_df['split'] = 'test'
    print(f"  [FALLBACK] Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    df['img_path'] = df.apply(lambda r: os.path.join(IMAGES_DIR, f"{r['dicom_id']}.jpg"), axis=1)
    has_img = df['img_path'].apply(os.path.exists).sum()
    print(f"  {name}: {has_img}/{len(df)} images accessible (all verified)")

# Copy checkpoints if available
ckpt_copied = 0
for search_dir in [PART1_DIR, '/kaggle/input', '/kaggle/working']:
    if os.path.exists(search_dir):
        for root, dirs, files in os.walk(search_dir):
            for f in files:
                if f.endswith('.pt'):
                    dst = os.path.join(Config.SAVE_DIR, f)
                    if not os.path.exists(dst):
                        try: shutil.copy2(os.path.join(root, f), dst); ckpt_copied += 1
                        except: pass
print(f"\n  [CHECKPOINTS] Copied {ckpt_copied} .pt files from previous parts")

# Find all_results.pkl from previous parts' outputs (in /kaggle/input/)
# and copy to /kaggle/working/ so the training loop can resume.
RESUME_PATH = '/kaggle/working/all_results.pkl'
if not os.path.exists(RESUME_PATH):
    for search_dir in ['/kaggle/input', '/kaggle/working']:
        if not os.path.exists(search_dir): continue
        for root, dirs, files in os.walk(search_dir):
            if root == '/kaggle/working': continue
            if 'all_results.pkl' in files:
                src = os.path.join(root, 'all_results.pkl')
                try:
                    shutil.copy2(src, RESUME_PATH)
                    print(f"  [COPY] {src} -> {RESUME_PATH}")
                    break
                except Exception as e:
                    print(f"  [ERR] copying all_results.pkl: {e}")
        if os.path.exists(RESUME_PATH): break

if os.path.exists(RESUME_PATH):
    with open(RESUME_PATH, 'rb') as f:
        all_results = pickle.load(f)
    print(f"  [RESUME] Loaded {len(all_results)} existing results")
else:
    all_results = {}

# Load baselines if available (trained in this Part 2 if missing)
img_text_path = os.path.join(PART1_DIR, 'img_text_results.pkl')
if not os.path.exists(img_text_path):
    img_text_path = '/kaggle/working/img_text_results.pkl'
if not os.path.exists(img_text_path):
    # Search /kaggle/input for previous parts' img_text_results.pkl
    for search_dir in ['/kaggle/input']:
        if not os.path.exists(search_dir): continue
        for root, dirs, files in os.walk(search_dir):
            if 'img_text_results.pkl' in files:
                src = os.path.join(root, 'img_text_results.pkl')
                dst = '/kaggle/working/img_text_results.pkl'
                try:
                    shutil.copy2(src, dst)
                    img_text_path = dst
                    print(f"  [COPY] {src} -> {dst}")
                    break
                except Exception as e:
                    print(f"  [ERR] copying img_text_results.pkl: {e}")
        if os.path.exists(img_text_path): break

if os.path.exists(img_text_path):
    with open(img_text_path, 'rb') as f:
        img_text_results = pickle.load(f)
    img_results = img_text_results.get('image_only', {})
    text_results = img_text_results.get('text_only', {})
    gray_results = img_text_results.get('gray_square', {})
    print(f"  [LOADED] img_text_results.pkl ({len(img_text_results)} baselines)")
else:
    img_results = {}; text_results = {}; gray_results = {}
    img_text_results = {}
    print(f"  [INFO] No baselines found - will train in this part")

print(f"\n  Part 1 output loaded. Ready for training.")


  LOADING PART 1 OUTPUT
  [FOUND] Part 1 output at: /kaggle/input/notebooks/tanvirmahmud13/medvision-thesis-part1
  [FOUND] Images at: /kaggle/input/notebooks/tanvirmahmud13/medvision-thesis-part1/images (2953 jpgs)
  [LOADED] mimic_df.pkl (2,962 rows)

  [VERIFY] Checking all 2,962 images...
  [WARN] 9 images missing! Removing from cohort and split...
  [WARN] Missing dicom_ids: ['2c1c13cf-af63e068-1491d5b2-757d9b43-5157b05c', '31edfe92-12f0e9e3-4f6351ce-f79b98dc-9d488336', '71b80518-2fc8f628-989069aa-29d55a34-22e69f97', '9547469b-8218bf88-fe063cf9-82881df6-a1eba5a9', '88cccea9-0d1ede71-a00d6379-7db37142-cf78a169', '4b5393dd-c1aa672f-d90c4e2e-e6c00cc6-03d6091c', '48d412b7-07495563-6ab1e000-abf9def9-f91fc4d9', 'd4d77dff-d915a19b-97e7ae99-51590919-7c2cd314', '3b6c8e23-115533bb-9d769fd7-ee3f53c9-d6013461']
  [CLEANED] mimic_df now has 2,953 rows (removed 9)

  [LOADING] Part 1's exact split from final_split.pkl
  [OK] Loaded cleaned split: Train=2049, Val=449, Test=455
  Train: 2049/2049

In [5]:
# Shared cells for Part 2 and Part 3: transforms, datasets, models, training function, OOF
# This file is included in both Part 2 and Part 3 notebooks

# Class weights
class_counts = np.array([train_df['label'].value_counts().get(i, 0) for i in range(3)])
beta = Config.CLASS_WEIGHT_BETA
effective_num = 1.0 - np.power(beta, class_counts)
class_weights_eff = (1.0 - beta) / np.maximum(effective_num, 1e-8)
class_weights_eff = class_weights_eff / class_weights_eff.sum() * len(class_counts)
CLASS_WEIGHTS = torch.FloatTensor(class_weights_eff).to(device)
print(f"Class counts: {class_counts.tolist()}")
print(f"Weights: N={class_weights_eff[0]:.3f}, PN={class_weights_eff[1]:.3f}, PNX={class_weights_eff[2]:.3f}")

# XRV normalization
class XRVNormalize:
    def __init__(self, maxval=255): self.maxval = maxval
    def __call__(self, img):
        img_array = np.array(img, dtype=np.float32)
        return torch.from_numpy(np.clip(img_array, 0, self.maxval) / self.maxval * 2 - 1).unsqueeze(0)

train_transform = transforms.Compose([
    transforms.Resize((256, 256)), transforms.RandomCrop(Config.IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5), transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Grayscale(num_output_channels=1), XRVNormalize(255),
])
train_transform_heavy = transforms.Compose([
    transforms.Resize((256, 256)), transforms.RandomCrop(Config.IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5), transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.5)),
    transforms.Grayscale(num_output_channels=1), XRVNormalize(255),
])
val_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1), XRVNormalize(255),
])
class AddGaussianNoise:
    def __init__(self, mean=0.0, std=0.25): self.mean = mean; self.std = std
    def __call__(self, tensor): return torch.clamp(torch.randn_like(tensor) * self.std + tensor, -1, 1)
noise_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1), XRVNormalize(255),
    AddGaussianNoise(0.0, 0.25),
])
print("Transforms defined (XRV normalization).")

# Datasets
def create_class_aware_augmented_df(df, pnx_mult=5, pn_mult=3):
    records = []
    for idx, row in df.iterrows():
        label = row['label']; base = {**row.to_dict(), 'orig_idx': idx}
        if label == 2:
            records.append({**base, 'aug_type': 'standard'})
            for i in range(pnx_mult - 1): records.append({**base, 'aug_type': f'heavy_{i}'})
        elif label == 1:
            records.append({**base, 'aug_type': 'standard'})
            for i in range(pn_mult - 1): records.append({**base, 'aug_type': f'moderate_{i}'})
        else: records.append({**base, 'aug_type': 'standard'})
    return pd.DataFrame(records)
def get_transform_for_aug_type(aug_type):
    if aug_type and aug_type.startswith('heavy'): return train_transform_heavy
    return train_transform
class ChestXrayDataset(Dataset):
    def __init__(self, df, tokenizer, transform=None, mask_text=False, stochastic_text_drop=0.0,
                 use_noise_image=False, use_gray_square=False, is_augmented=False, return_index=False):
        self.df = df.reset_index(drop=True); self.tokenizer = tokenizer; self.transform = transform
        self.mask_text = mask_text; self.stochastic_text_drop = stochastic_text_drop
        self.use_noise_image = use_noise_image; self.use_gray_square = use_gray_square
        self.is_augmented = is_augmented; self.return_index = return_index
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.use_gray_square:
            img = Image.new('L', (Config.IMG_SIZE, Config.IMG_SIZE), color=128)
            img_tensor = XRVNormalize(255)(img)
        else:
            try:
                img = Image.open(row['img_path']).convert('RGB'); img.load()
            except Exception as e:
                if 'truncated' in str(e).lower() or 'image file' in str(e).lower():
                    img = Image.new('RGB', (Config.IMG_SIZE, Config.IMG_SIZE), color=128)
                else: raise RuntimeError(f"Image error: {row['img_path']}. {e}") from e
            if self.use_noise_image: img_tensor = noise_transform(img)
            elif self.is_augmented and self.transform is None:
                img_tensor = get_transform_for_aug_type(row.get('aug_type', 'standard'))(img)
            else: img_tensor = self.transform(img) if self.transform else val_transform(img)
        text = row['findings']
        if self.mask_text: text = ''
        if self.stochastic_text_drop > 0 and random.random() < self.stochastic_text_drop: text = ''
        enc = self.tokenizer(text, max_length=Config.TEXT_MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')
        label = torch.tensor(row['label'], dtype=torch.long)
        if self.return_index:
            return (img_tensor, enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0), label, idx)
        return (img_tensor, enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0), label)
class ImageOnlyDataset(Dataset):
    def __init__(self, df, transform=None, is_augmented=False):
        self.df = df.reset_index(drop=True); self.transform = transform or val_transform; self.is_augmented = is_augmented
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try: img = Image.open(row['img_path']).convert('RGB'); img.load()
        except Exception as e:
            if 'truncated' in str(e).lower() or 'image file' in str(e).lower():
                img = Image.new('RGB', (Config.IMG_SIZE, Config.IMG_SIZE), color=128)
            else: raise RuntimeError(f"Image error: {row['img_path']}. {e}") from e
        if self.is_augmented: img_tensor = get_transform_for_aug_type(row.get('aug_type', 'standard'))(img)
        else: img_tensor = self.transform(img)
        return (img_tensor, torch.tensor(row['label'], dtype=torch.long))
class TextOnlyDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df.reset_index(drop=True); self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(row['findings'], max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return (enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0), torch.tensor(row['label'], dtype=torch.long))
def make_weighted_sampler(df, pnx_per_batch=2, batch_size=8):
    labels = df['label'].values; class_counts = np.bincount(labels, minlength=3)
    weights = np.zeros(3, dtype=np.float64)
    for c in range(3):
        if class_counts[c] > 0:
            if c == 2: weights[c] = pnx_per_batch / max(class_counts[c], 1)
            else: weights[c] = (batch_size - pnx_per_batch) / 2.0 / max(class_counts[c], 1)
    weights = weights / weights.sum()
    sw = np.array([weights[l] for l in labels]); sw = sw / sw.sum() * len(sw)
    return WeightedRandomSampler(sw, num_samples=len(sw), replacement=True)
tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_ENCODER)
train_df_aug = create_class_aware_augmented_df(train_df, pnx_mult=Config.TB_OVERSAMPLE, pn_mult=Config.PN_OVERSAMPLE)
print(f"Train (augmented): {len(train_df_aug)} (was {len(train_df)})")
train_sampler = make_weighted_sampler(train_df_aug, pnx_per_batch=Config.TB_PER_BATCH, batch_size=Config.BATCH_SIZE)
train_ds = ChestXrayDataset(train_df_aug, tokenizer, is_augmented=True, stochastic_text_drop=Config.STOCHASTIC_TEXT_DROP, return_index=True)
val_ds = ChestXrayDataset(val_df, tokenizer, transform=val_transform)
test_ds = ChestXrayDataset(test_df, tokenizer, transform=val_transform)
test_noise_ds = ChestXrayDataset(test_df, tokenizer, transform=val_transform, use_noise_image=True)
loaders = {'vlm': {
    'train': DataLoader(train_ds, batch_size=Config.BATCH_SIZE, sampler=train_sampler, num_workers=0, pin_memory=True, drop_last=True),
    'val': DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    'test': DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    'test_noise': DataLoader(test_noise_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
}}
print(f"Dataloaders ready. Train batches: {len(loaders['vlm']['train'])}")
mem_stats("after dataloaders")


Class counts: [1106, 738, 205]
Weights: N=0.953, PN=0.954, PNX=1.093
Transforms defined (XRV normalization).
Train (augmented): 4345 (was 2049)
Dataloaders ready. Train batches: 543
  [mem after dataloaders] RAM=1071MB, GPU=0MB


In [6]:
# Model architecture (all fusion types)
class XRVImageEncoder(nn.Module):
    def __init__(self, proj_dim=512, freeze_stem=True):
        super().__init__()
        self.backbone = xrv.models.DenseNet(weights='densenet121-res224-mimic_ch')
        num_features = self.backbone.classifier.in_features
        if freeze_stem:
            for name, p in self.backbone.named_parameters():
                if any(name.startswith(prefix) for prefix in ['features.conv0', 'features.norm0', 'features.pool0', 'features.denseblock1', 'features.transition1']): p.requires_grad = False
        self.projection = nn.Sequential(nn.Linear(num_features, proj_dim), nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(0.25))
    def forward(self, x): return self.projection(self.backbone.features2(x))
class TextEncoder(nn.Module):
    def __init__(self, model_name='emilyalsentzer/Bio_ClinicalBERT', proj_dim=512, frozen_layers=4):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for i in range(min(frozen_layers, len(self.bert.encoder.layer))):
            for p in self.bert.encoder.layer[i].parameters(): p.requires_grad = False
        self.projection = nn.Sequential(nn.Linear(self.bert.config.hidden_size, proj_dim), nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(0.25))
    def forward(self, input_ids, attention_mask):
        return self.projection(self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :])
class DualPathCADQ(nn.Module):
    def __init__(self, proj_dim=512, num_classes=3, num_heads=8, dropout=0.25, min_gate=0.20, max_gate=0.80, gate_init_logits=None):
        super().__init__()
        self.min_gate = min_gate; self.max_gate = max_gate
        self.cross_attn = nn.MultiheadAttention(embed_dim=proj_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(proj_dim); self.img_head = nn.Linear(proj_dim, num_classes); self.text_head = nn.Linear(proj_dim, num_classes)
        self.gate_logits = nn.Parameter(torch.tensor(gate_init_logits or [-0.5, 0.0, 0.5], dtype=torch.float32))
        self.has_gate = True
    def get_alpha(self): return self.min_gate + (self.max_gate - self.min_gate) * torch.sigmoid(self.gate_logits)
    def forward(self, img_feat, text_feat, p_bias=None):
        img_seq = img_feat.unsqueeze(1); text_seq = text_feat.unsqueeze(1)
        attn_out, _ = self.cross_attn(query=img_seq, key=text_seq, value=text_seq)
        img_enriched = self.norm(attn_out.squeeze(1) + img_feat)
        alpha = self.get_alpha()
        logits = alpha.unsqueeze(0) * self.img_head(img_enriched) + (1 - alpha.unsqueeze(0)) * self.text_head(text_feat)
        return logits, alpha
class LearnedMixinFusion(nn.Module):
    def __init__(self, proj_dim=512, num_classes=3, num_heads=8, dropout=0.25, g_max=5.0, g_init_bias=-2.0):
        super().__init__(); self.g_max = g_max
        self.cross_attn = nn.MultiheadAttention(embed_dim=proj_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(proj_dim); self.img_head = nn.Linear(proj_dim, num_classes); self.text_head = nn.Linear(proj_dim, num_classes)
        self.g_head = nn.Linear(proj_dim * 2, 1); nn.init.zeros_(self.g_head.weight); nn.init.constant_(self.g_head.bias, g_init_bias)
        self.has_gate = False
    def forward(self, img_feat, text_feat, log_p_bias=None):
        img_seq = img_feat.unsqueeze(1); text_seq = text_feat.unsqueeze(1)
        attn_out, _ = self.cross_attn(query=img_seq, key=text_seq, value=text_seq)
        img_enriched = self.norm(attn_out.squeeze(1) + img_feat)
        main_logits = 0.5 * self.img_head(img_enriched) + 0.5 * self.text_head(text_feat)
        if log_p_bias is not None:
            h = torch.cat([img_enriched, text_feat], dim=-1).detach()
            g = self.g_max * torch.sigmoid(self.g_head(h)).squeeze(-1)
            return F.log_softmax(main_logits, dim=-1) + g.unsqueeze(-1) * log_p_bias, g
        return main_logits, torch.zeros(main_logits.size(0), device=main_logits.device)
    def compute_entropy_penalty(self, g, log_p_bias):
        bl = g.unsqueeze(-1) * log_p_bias; bp = F.softmax(bl, dim=-1); lb = F.log_softmax(bl, dim=-1)
        return -(bp * lb).sum(dim=-1).mean()
class NoiseConsistencyFusion(nn.Module):
    def __init__(self, proj_dim=512, num_classes=3, num_heads=8, dropout=0.25):
        super().__init__(); self.cross_attn = nn.MultiheadAttention(embed_dim=proj_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(proj_dim); self.img_head = nn.Linear(proj_dim, num_classes); self.text_head = nn.Linear(proj_dim, num_classes)
        self.has_gate = False
    def forward(self, img_feat, text_feat, p_bias=None):
        img_seq = img_feat.unsqueeze(1); text_seq = text_feat.unsqueeze(1)
        attn_out, _ = self.cross_attn(query=img_seq, key=text_seq, value=text_seq)
        img_enriched = self.norm(attn_out.squeeze(1) + img_feat)
        return 0.5 * self.img_head(img_enriched) + 0.5 * self.text_head(text_feat), torch.zeros(img_feat.size(0), device=img_feat.device)
    def compute_consistency_loss(self, model, images, input_ids, attention_mask, device):
        with autocast('cuda'):
            logits_real, _ = model(images, input_ids, attention_mask)
            p_real = F.softmax(logits_real.float(), dim=-1)
            noise = torch.randn_like(images) * 0.25; noise_images = torch.clamp(images + noise, -1.0, 1.0)
            logits_noised, _ = model(noise_images, input_ids, attention_mask)
            p_noised = F.softmax(logits_noised.float(), dim=-1)
            kl = F.kl_div(torch.log(p_noised + 1e-8), p_real, reduction='none').sum(dim=-1)
            return -kl.mean()
class AdaptiveCADQ(nn.Module):
    def __init__(self, proj_dim=512, num_classes=3, num_heads=8, dropout=0.25):
        super().__init__(); self.cross_attn = nn.MultiheadAttention(embed_dim=proj_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(proj_dim); self.img_head = nn.Linear(proj_dim, num_classes); self.text_head = nn.Linear(proj_dim, num_classes)
        self.gate_mlp = nn.Sequential(nn.Linear(2, 16), nn.GELU(), nn.Dropout(dropout), nn.Linear(16, 1), nn.Sigmoid())
        self.has_gate = True
    def get_alpha(self, img_logits=None, text_logits=None):
        if img_logits is None or text_logits is None: return getattr(self, '_last_alpha', torch.tensor([0.5, 0.5, 0.5]))
        pi = F.softmax(img_logits, dim=-1); pt = F.softmax(text_logits, dim=-1)
        ei = -(pi * torch.log(pi + 1e-8)).sum(dim=-1, keepdim=True); et = -(pt * torch.log(pt + 1e-8)).sum(dim=-1, keepdim=True)
        a = self.gate_mlp(torch.cat([ei, et], dim=-1)).squeeze(-1); self._last_alpha = a.detach(); return a
    def forward(self, img_feat, text_feat, p_bias=None):
        img_seq = img_feat.unsqueeze(1); text_seq = text_feat.unsqueeze(1)
        attn_out, _ = self.cross_attn(query=img_seq, key=text_seq, value=text_seq)
        ie = self.norm(attn_out.squeeze(1) + img_feat); il = self.img_head(ie); tl = self.text_head(text_feat)
        a = self.get_alpha(il, tl); return a.unsqueeze(-1) * il + (1 - a.unsqueeze(-1)) * tl, a
class ChestXrayVLM(nn.Module):
    def __init__(self, config, fusion_type='cadq'):
        super().__init__()
        self.image_encoder = XRVImageEncoder(proj_dim=config.PROJECTION_DIM, freeze_stem=config.IMAGE_FROZEN_STEM_ONLY)
        self.text_encoder = TextEncoder(model_name=config.TEXT_ENCODER, proj_dim=config.PROJECTION_DIM, frozen_layers=config.TEXT_FROZEN_LAYERS)
        ft_map = {'cadq': lambda: DualPathCADQ(proj_dim=config.PROJECTION_DIM, num_classes=config.NUM_CLASSES, num_heads=config.NUM_HEADS, dropout=config.DROPOUT_RATE, min_gate=config.CADQ_MIN_IMAGE_GATE, max_gate=config.CADQ_MAX_IMAGE_GATE, gate_init_logits=config.GATE_INIT_LOGITS),
                  'learned_mixin': lambda: LearnedMixinFusion(proj_dim=config.PROJECTION_DIM, num_classes=config.NUM_CLASSES, num_heads=config.NUM_HEADS, dropout=config.DROPOUT_RATE, g_max=config.LMH_G_MAX, g_init_bias=config.LMH_G_INIT_BIAS),
                  'noise_consistency': lambda: NoiseConsistencyFusion(proj_dim=config.PROJECTION_DIM, num_classes=config.NUM_CLASSES, num_heads=config.NUM_HEADS, dropout=config.DROPOUT_RATE),
                  'adaptive_cadq': lambda: AdaptiveCADQ(proj_dim=config.PROJECTION_DIM, num_classes=config.NUM_CLASSES, num_heads=config.NUM_HEADS, dropout=config.DROPOUT_RATE)}
        if fusion_type not in ft_map: raise ValueError(f"Unknown fusion_type: {fusion_type}")
        self.fusion = ft_map[fusion_type](); self.fusion_type = fusion_type
    def forward(self, images, input_ids, attention_mask, log_p_bias=None):
        img_feat = self.image_encoder(images); text_feat = self.text_encoder(input_ids, attention_mask)
        if self.fusion_type == 'learned_mixin': return self.fusion(img_feat, text_feat, log_p_bias=log_p_bias)
        return self.fusion(img_feat, text_feat)
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5, label_smoothing=0.05):
        super().__init__(); self.gamma = gamma; self.label_smoothing = label_smoothing
        if alpha is not None: self.register_buffer('alpha', alpha.float())
        else: self.alpha = None
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce); return (((1 - pt) ** self.gamma) * ce).mean()
class ImageOnlyModel(nn.Module):
    def __init__(self, config):
        super().__init__(); self.encoder = XRVImageEncoder(proj_dim=config.PROJECTION_DIM, freeze_stem=config.IMAGE_FROZEN_STEM_ONLY)
        self.classifier = nn.Sequential(nn.Linear(config.PROJECTION_DIM, config.PROJECTION_DIM // 2), nn.GELU(), nn.Dropout(0.3), nn.Linear(config.PROJECTION_DIM // 2, config.NUM_CLASSES))
    def forward(self, x): return self.classifier(self.encoder(x))
class TextOnlyModel(nn.Module):
    def __init__(self, config):
        super().__init__(); self.bert = AutoModel.from_pretrained(config.TEXT_ENCODER)
        for p in self.bert.parameters(): p.requires_grad = False
        self.head = nn.Sequential(nn.Linear(self.bert.config.hidden_size, 256), nn.GELU(), nn.Dropout(0.3), nn.Linear(256, config.NUM_CLASSES))
    def forward(self, input_ids, attention_mask):
        return self.head(self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :])
print("Model classes defined.")


Model classes defined.


In [7]:
# Training function + evaluate_model + OOF bias model
def evaluate_model(model, loader, model_type='vlm', device=None):
    if device is None: device = next(model.parameters()).device
    model.eval(); all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Eval', leave=False):
            if model_type == 'vlm':
                if len(batch) == 5: images, input_ids, attention_mask, labels, idx = [b.to(device) for b in batch]
                else: images, input_ids, attention_mask, labels = [b.to(device) for b in batch]
                with autocast('cuda'): logits, _ = model(images, input_ids, attention_mask, log_p_bias=None)
            elif model_type == 'image_only':
                images, labels = [b.to(device) for b in batch]
                with autocast('cuda'): logits = model(images)
            elif model_type == 'text_only':
                input_ids, attention_mask, labels = [b.to(device) for b in batch]
                with autocast('cuda'): logits = model(input_ids, attention_mask)
            probs = F.softmax(logits.float(), dim=-1); preds = logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().long().numpy()); all_labels.extend(labels.cpu().long().numpy())
            all_probs.extend(probs.cpu().float().numpy())
    all_preds = np.array(all_preds); all_labels = np.array(all_labels); all_probs = np.array(all_probs)
    try: auc = roc_auc_score(all_labels, all_probs / np.maximum(all_probs.sum(axis=1, keepdims=True), 1e-8), multi_class='ovr', average='macro')
    except: auc = 0.0
    class_report = classification_report(all_labels, all_preds, target_names=Config.LABEL_NAMES, output_dict=True, zero_division=0)
    return {'accuracy': accuracy_score(all_labels, all_preds), 'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0),
            'auc_ovr': auc, 'per_class_f1': f1_score(all_labels, all_preds, average=None, zero_division=0),
            'per_class_recall': recall_score(all_labels, all_preds, average=None, zero_division=0),
            'all_preds': all_preds, 'all_labels': all_labels, 'all_probs': all_probs,
            'confusion_matrix': confusion_matrix(all_labels, all_preds).tolist(), 'classification_report': class_report}

def train_vlm(model, loaders, config, model_name='vlm', epochs=None, warm_start_path=None,
              extra_loss_fn=None, extra_loss_weight=0.0, p_bias_lookup=None, lmh_lambda_entropy=0.0):
    if epochs is None: epochs = config.BASELINE_EPOCHS
    model = model.to(device)
    if warm_start_path is not None and os.path.exists(warm_start_path):
        print(f"  Warm-starting from {warm_start_path}")
        baseline_state = torch.load(warm_start_path, map_location=device)
        model_state = model.state_dict(); loaded = 0; skipped = 0
        for name, param in baseline_state.items():
            if name in model_state and model_state[name].shape == param.shape: model_state[name] = param; loaded += 1
            else: skipped += 1
        model.load_state_dict(model_state); print(f"  Loaded {loaded} params, skipped {skipped}")
    param_groups = [{'params': [p for p in model.image_encoder.parameters() if p.requires_grad], 'lr': config.LR_IMG_BACKBONE},
                    {'params': [p for p in model.text_encoder.parameters() if p.requires_grad], 'lr': config.LR_TEXT}]
    fusion_params = []; gate_params = []; g_head_params = []
    for name, p in model.fusion.named_parameters():
        if not p.requires_grad: continue
        if name == 'gate_logits': gate_params.append(p)
        elif name.startswith('g_head'): g_head_params.append(p)
        else: fusion_params.append(p)
    if fusion_params: param_groups.append({'params': fusion_params, 'lr': config.LR_FUSION})
    if gate_params: param_groups.append({'params': gate_params, 'lr': config.LR_GATE})
    if g_head_params: param_groups.append({'params': g_head_params, 'lr': getattr(config, 'LR_G_HEAD', config.LR_FUSION)})
    optimizer = torch.optim.AdamW(param_groups, weight_decay=config.WEIGHT_DECAY)
    max_lrs = [g['lr'] for g in param_groups]
    total_steps = len(loaders['train']) * epochs // config.GRAD_ACCUM_STEPS
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=max_lrs, total_steps=total_steps, pct_start=config.WARMUP_RATIO)
    scaler = GradScaler('cuda')
    criterion = FocalLoss(alpha=CLASS_WEIGHTS, gamma=config.FOCAL_GAMMA, label_smoothing=config.LABEL_SMOOTHING)
    is_lmh = hasattr(model, 'fusion_type') and model.fusion_type == 'learned_mixin'
    history = {'train_loss': [], 'val_f1': [], 'val_pnx_recall': []}
    g_sum = 0.0
    best_val_score = -1.0; best_state = None
    for epoch in range(epochs):
        model.train(); optimizer.zero_grad(); epoch_loss = 0.0; n_batches = 0
        pbar = tqdm(loaders['train'], desc=f'[{model_name}] E{epoch+1}/{epochs}', leave=False)
        for step, batch in enumerate(pbar):
            if len(batch) == 5: images, input_ids, attention_mask, labels, idx = [b.to(device) for b in batch]
            else: images, input_ids, attention_mask, labels = [b.to(device) for b in batch]; idx = None
            log_p_bias = None
            if is_lmh and p_bias_lookup is not None and idx is not None:
                _, raw_probs = p_bias_lookup.get(idx.cpu().numpy())
                log_p_bias = torch.log(torch.from_numpy(raw_probs).float().to(device).clamp(min=1e-8))
            with autocast('cuda'):
                logits, aux = model(images, input_ids, attention_mask, log_p_bias=log_p_bias if is_lmh else None)
                loss_ce = criterion(logits, labels)
                if is_lmh and log_p_bias is not None and lmh_lambda_entropy > 0:
                    g = aux; entropy_pen = model.fusion.compute_entropy_penalty(g, log_p_bias)
                    anneal = getattr(config, "LMH_ANNEAL_EPOCHS", 3)
                    current_lambda = lmh_lambda_entropy * min(1.0, (epoch + 1) / anneal) if epoch < anneal else lmh_lambda_entropy
                    loss = loss_ce + current_lambda * entropy_pen; g_sum += float(g.mean().item())
                elif extra_loss_fn is not None and extra_loss_weight > 0:
                    loss_extra = extra_loss_fn(logits=logits, alpha=aux, images=images, input_ids=input_ids, attention_mask=attention_mask, labels=labels, model=model, indices=idx)
                    loss = loss_ce + extra_loss_weight * loss_extra
                else: loss = loss_ce
            scaler.scale(loss / config.GRAD_ACCUM_STEPS).backward()
            if (step + 1) % config.GRAD_ACCUM_STEPS == 0:
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(); scheduler.step()
            epoch_loss += float(loss.item()); n_batches += 1; pbar.set_postfix(loss=f'{loss.item():.4f}')
        val_m = evaluate_model(model, loaders['val'], model_type='vlm')
        val_f1 = val_m['macro_f1']; val_pnx = val_m['per_class_recall'][2] if len(val_m['per_class_recall']) > 2 else 0.0
        history['train_loss'].append(epoch_loss / max(n_batches, 1)); history['val_f1'].append(val_f1); history['val_pnx_recall'].append(val_pnx)
        extra = f', g_mean={g_sum/max(n_batches,1):.4f}' if is_lmh else ''
        print(f"  E{epoch+1}: loss={epoch_loss/max(n_batches,1):.4f}, val_f1={val_f1:.4f}, val_pnx={val_pnx:.4f}{extra}")
        if val_f1 > best_val_score: best_val_score = val_f1; best_state = copy.deepcopy(model.state_dict())
    if best_state is None: best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, history

# OOF bias model
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(train_df['findings'].fillna(''))
y_train_lr = train_df['label'].values
print("  Computing OOF LR-TF-IDF...")
skf = StratifiedKFold(n_splits=Config.LR_OOF_FOLDS, shuffle=True, random_state=42)
lr_oof = LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs', C=1.0, random_state=42)
oof_preds = cross_val_predict(lr_oof, X_train_tfidf, y_train_lr, cv=skf, method='predict')
oof_probs = cross_val_predict(lr_oof, X_train_tfidf, y_train_lr, cv=skf, method='predict_proba')
print(f"    OOF accuracy: {accuracy_score(y_train_lr, oof_preds):.4f}")
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs', C=1.0, random_state=42)
lr_model.fit(X_train_tfidf, y_train_lr)
from sklearn.metrics import log_loss
if Config.LMH_CALIBRATE_BIAS:
    X_val_cal = vectorizer.transform(val_df['findings'].fillna(''))
    val_probs_raw = lr_model.predict_proba(X_val_cal); val_labels_cal = val_df['label'].values
    best_T, best_loss = 1.0, float('inf')
    for T_try in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0]:
        cal = np.power(val_probs_raw, 1.0 / T_try); cal = cal / cal.sum(axis=1, keepdims=True)
        try:
            ll = log_loss(val_labels_cal, cal, labels=[0,1,2])
            if ll < best_loss: best_loss = ll; best_T = T_try
        except: pass
    T = best_T; print(f"    Temperature: T={T}")
else: T = Config.LMH_BIAS_TEMP
oof_probs_cal = np.power(oof_probs, 1.0 / T); oof_probs_cal = oof_probs_cal / oof_probs_cal.sum(axis=1, keepdims=True)
epsilon = Config.LMH_BIAS_EPSILON; uniform = np.ones_like(oof_probs_cal) / oof_probs_cal.shape[1]
oof_probs_smoothed = (1 - epsilon) * oof_probs_cal + epsilon * uniform
aug_oof_probs = oof_probs_smoothed[train_df_aug['orig_idx'].values]
class TextBaselineLookup:
    def __init__(self, probs): self.probs = probs
    def get(self, indices): return None, self.probs[indices]
p_bias_lookup = TextBaselineLookup(aug_oof_probs)
def gacr_b_loss(logits, labels, text_baseline_preds, text_baseline_probs):
    p_full = F.softmax(logits, dim=-1); tb_preds = text_baseline_preds.to(logits.device)
    tb_soft = torch.from_numpy(text_baseline_probs).float().to(logits.device)
    sim = F.cosine_similarity(p_full, tb_soft, dim=-1); mask = (tb_preds != labels).float()
    return (mask * (1.0 - sim)).mean()
aug_oof_preds_raw = oof_preds[train_df_aug['orig_idx'].values]; aug_oof_probs_raw = oof_probs[train_df_aug['orig_idx'].values]
class GACRLookup:
    def __init__(self, preds, probs): self.preds = preds; self.probs = probs
    def get(self, indices): return self.preds[indices], self.probs[indices]
gacr_lookup = GACRLookup(aug_oof_preds_raw, aug_oof_probs_raw)
print("  OOF bias model + GACR-B ready")


  Computing OOF LR-TF-IDF...
    OOF accuracy: 0.9195
    Temperature: T=0.5
  OOF bias model + GACR-B ready


In [8]:
# === Part 10: GACR-B ablation training (3 seeds) ===
print("=" * 70)
print("  PART 10: GACR-B ABLATION TRAINING")
print("=" * 70)
print(f"  Targets: 3 GACR-B seeds = {Config.ABLATION_SEED_LIST}")

# GACR-B requires the gacr_lookup table (built in setup cell 5/7)
# It also requires baseline checkpoints for warm-start if Config.LMH_WARM_START is True
# — but GACR-B does not use warm-start, so we don't need them.

# ─── GACR-B (3 seeds) ─────────────────────────────────────────
for seed in Config.ABLATION_SEED_LIST:
    key = f'gacr_B_s{seed}'
    if key in all_results:
        print(f"  [SKIP] {key} (F1={all_results[key]['test']['macro_f1']:.4f})")
        continue
    print(f"\n{'='*70}\n  GACR-B -- SEED {seed}\n{'='*70}")
    set_seed(seed)
    m = ChestXrayVLM(Config, fusion_type='cadq').to(device)
    def gacr_b_extra(logits, alpha, images, input_ids, attention_mask, labels, model, indices=None, **kw):
        if indices is None: return torch.tensor(0.0, device=logits.device)
        tb_preds, tb_probs = gacr_lookup.get(indices.cpu().numpy())
        return gacr_b_loss(logits, labels, torch.from_numpy(tb_preds).long().to(logits.device), tb_probs)
    m, h = train_vlm(m, loaders['vlm'], Config, model_name=f'gacrB_s{seed}', epochs=Config.GACR_B_EPOCHS, extra_loss_fn=gacr_b_extra, extra_loss_weight=0.3)
    t = evaluate_model(m, loaders['vlm']['test'], model_type='vlm')
    print(f"  GACR-B s{seed}: F1={t['macro_f1']:.4f}, PNX-F1={t['per_class_f1'][2]:.4f}")
    all_results[key] = {'history': h, 'test': t}
    torch.save({k: v.half() if v.dtype == torch.float32 else v for k, v in m.state_dict().items()}, os.path.join(Config.SAVE_DIR, f'gacr_b_s{seed}.pt'))
    del m; gc.collect(); torch.cuda.empty_cache()
    with open(RESUME_PATH, 'wb') as f: pickle.dump(all_results, f)

# Summary
print(f"\n{'='*70}\n  PART 10 TRAINING SUMMARY\n{'='*70}")
import numpy as _np
gacr_f1s = [all_results[k]['test']['macro_f1'] for k in sorted(all_results) if k.startswith('gacr_B_s')]
if gacr_f1s:
    print(f"  GACR-B: {len(gacr_f1s)} seeds trained, F1={_np.mean(gacr_f1s):.4f}+/-{_np.std(gacr_f1s, ddof=1) if len(gacr_f1s) > 1 else 0:.4f}")
total = len([k for k in all_results if k.startswith('gacr_B_s')])
print(f"\n  Total GACR-B seeds done across all parts: {total}/3")
print(f"\n  NEXT: Save Version > Quick Save, then run Final Part (Adaptive-CADQ + figures)")


  PART 10: GACR-B ABLATION TRAINING
  Targets: 3 GACR-B seeds = [42, 123, 456]

  GACR-B -- SEED 42
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/mimic_ch-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/mimic_ch-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]


[gacrB_s42] E1/10:   0%|          | 0/543 [00:00<?, ?it/s]

Eval:   0%|          | 0/57 [00:00<?, ?it/s]

  E1: loss=0.3875, val_f1=0.8721, val_pnx=0.9688


[gacrB_s42] E2/10:   0%|          | 0/543 [00:00<?, ?it/s]

...

[gacrB_s456] E10/10:   0%|          | 0/543 [00:00<?, ?it/s]

Eval:   0%|          | 0/57 [00:00<?, ?it/s]

  E10: loss=0.0795, val_f1=0.9229, val_pnx=0.9062


Eval:   0%|          | 0/57 [00:00<?, ?it/s]

  GACR-B s456: F1=0.9498, PNX-F1=0.9032

  PART 10 TRAINING SUMMARY
  GACR-B: 3 seeds trained, F1=0.9351+/-0.0130

  Total GACR-B seeds done across all parts: 3/3

  NEXT: Save Version > Quick Save, then run Final Part (Adaptive-CADQ + figures)


In [9]:
# Save all Part 10 artifacts
print("=" * 70)
print(f"  PART 10 FINAL SAVE")
print("=" * 70)
with open('/kaggle/working/all_results.pkl', 'wb') as f: pickle.dump(all_results, f)
print(f"  [SAVED] all_results.pkl ({len(all_results)} entries)")
with open('/kaggle/working/img_text_results.pkl', 'wb') as f: pickle.dump(img_text_results, f)
print(f"  [SAVED] img_text_results.pkl ({len(img_text_results)} baselines)")
ckpt_files = [f for f in os.listdir(Config.SAVE_DIR) if f.endswith('.pt')]
print(f"  [CHECKPOINTS] {len(ckpt_files)} .pt files in Config.SAVE_DIR")
print(f"\n  Trained in this part:")
for k in sorted(all_results.keys()):
    if k.startswith('gacr_B_s') or k.startswith('adaptive_cadq_s'):
        print(f"    {k}: F1={all_results[k]['test']['macro_f1']:.4f}")
print(f"\n======================================================================\n  PART 10 COMPLETE\n======================================================================")
print(f"  NEXT STEPS:")
print(f"  1. Save Version > Quick Save (top right)")
print(f"  2. Open medvision-thesis-final.ipynb → Run All")


  PART 10 FINAL SAVE
  [SAVED] all_results.pkl (27 entries)
  [SAVED] img_text_results.pkl (3 baselines)
  [CHECKPOINTS] 27 .pt files in Config.SAVE_DIR

  Trained in this part:
    gacr_B_s123: F1=0.9307
    gacr_B_s42: F1=0.9248
    gacr_B_s456: F1=0.9498

  PART 10 COMPLETE
  NEXT STEPS:
  1. Save Version > Quick Save (top right)
  2. Open medvision-thesis-final.ipynb → Run All
